# Road IDS Models and Table II

**Score Calculator**

In [111]:
def compute_metrics(model_name, y_test, y_pred):
    f1 = f1_score(y_test, y_pred, zero_division=0)
    acc = accuracy_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    mcc = matthews_corrcoef(y_test, y_pred)
    
    benign_count = int((y_test == 0).sum())     # = tn + fp
    malicious_count = int((y_test == 1).sum())  # = tp + fn

    result = [model_name, benign_count, malicious_count, f1, fp,fn, mcc]
    return result

**Shallow Models Run Function**

In [ ]:
def evaluate_shallow_model(name, model, X_train, y_train, X_test, y_test, save_path=None):
    """
    Train a model, predict multiclass, collapse to binary (0=normal, 1=attack),
    and return evaluation metrics.
    Need to be careful about the versions: Python=3.10, tensorflow=2.12.0, Numpy=1.23, art=1.16 and Scikit learn version=1.3.2
    """
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)    # ----- Predict (multiclass) -----
    y_test = (y_test != 0).astype(int)     # ----- Collapse to binary -----
    y_pred = (y_pred != 0).astype(int)

    result = compute_metrics(name, y_test, y_pred)
    print(name)
    if save_path: joblib.dump(model, f"{save_path}/{name.lower()}_model.pkl")
    return result

**DNN Model Run Function**

In [114]:
def dnn_model_run(X_train = X_train, X_test = X_test, y_train = y_train, y_test = y_test, 
                  bs = 32, split = .1, epo = 5, vs = .1, vb = 1, pt= 3, pred=.5, pred_bs=1024):
    
    model = Sequential([   
    Input(shape=(X_train.shape[1],)),  # Input layer (should be 10 features)
        Dense(16, activation='relu'),#Dropout(0.3),
        Dense(16, activation='relu'),#Dropout(0.3),
        Dense(16, activation='relu'),#Dropout(0.3),
        Dense(16, activation='relu'),#Dropout(0.3),
        Dense(1, activation='sigmoid')  # Output layer: 1 neurons
    ])  # Rebuild model from scratch
    
    # model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    model.compile(optimizer=legacy_optimizers.Adam(), loss='binary_crossentropy', metrics=['accuracy'])
    
    # Early stopping callback
    early_stop = EarlyStopping(monitor='loss', patience=pt, restore_best_weights=True)
    
    history = model.fit(
        X_train, y_train, 
        validation_split= vs, 
        epochs=epo, 
        batch_size=bs, 
        verbose=vb,
        callbacks = [early_stop]
    )
    
    # ----- Predict -----
    y_pred_prob = model.predict(X_test, batch_size= pred_bs)
    y_pred = (y_pred_prob > pred).astype(int)
    
    # ----- Evaluation -----
    # Convert multiclass to binary: 0 = normal, 1 = any attack
    # y_test = (y_test != 0).astype(int)
    # y_pred = (y_pred != 0).astype(int)
    
    return model, y_pred


## Table II Generation

**Train - Test Split**

In [128]:

import pandas as pd
from sklearn.model_selection import train_test_split

path_result = "results"
df=pd.read_csv(f"{path_result}/attack_data.csv")
X = df.drop(columns=['Flag'], errors='ignore')
y = df['Flag']

# Stratified split Train/Test (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, random_state=42, stratify=y
)
X_train = X_train.copy()
X_test = X_test.copy()
X_train.columns = [c.replace("[", "_").replace("]", "").replace("<", "_") for c in X_train.columns]
X_test.columns  = [c.replace("[", "_").replace("]", "").replace("<", "_") for c in X_test.columns]

print("Train samples:", len(X_train), "Test samples:", len(X_test), "Total samples: ", len(X_test)+len(X_train))
# print(X_train.head())

Train samples: 1086055 Test samples: 465453 Total samples:  1551508


**DNN Model**

In [116]:
import math
from sklearn.metrics import matthews_corrcoef, f1_score, confusion_matrix, accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import legacy as legacy_optimizers

model_name = "DNN"; bs = 32; split = .1; epo = 50; vs = .1; vb = 1; pt=3
dnn_model, y_pred = dnn_model_run(X_train, X_test, y_train, y_test, bs, split, epo, vs, vb, pt, pred=.5, pred_bs=1024)
result_dnn = compute_metrics(model_name, y_test, y_pred)


Epoch 1/50
30546/30546 [==============================] - 155s 5ms/step - loss: 0.1436 - accuracy: 0.9804 - val_loss: 0.0404 - val_accuracy: 0.9874
Epoch 2/50
30546/30546 [==============================] - 152s 5ms/step - loss: 0.0405 - accuracy: 0.9868 - val_loss: 0.0346 - val_accuracy: 0.9887
Epoch 3/50
30546/30546 [==============================] - 149s 5ms/step - loss: 0.0378 - accuracy: 0.9875 - val_loss: 0.0363 - val_accuracy: 0.9886
Epoch 4/50
30546/30546 [==============================] - 159s 5ms/step - loss: 0.0359 - accuracy: 0.9880 - val_loss: 0.0339 - val_accuracy: 0.9884
Epoch 5/50
30546/30546 [==============================] - 145s 5ms/step - loss: 0.0354 - accuracy: 0.9881 - val_loss: 0.0344 - val_accuracy: 0.9889
Epoch 6/50
30546/30546 [==============================] - 143s 5ms/step - loss: 0.0344 - accuracy: 0.9884 - val_loss: 0.0382 - val_accuracy: 0.9876
Epoch 7/50
30546/30546 [==============================] - 142s 5ms/step - loss: 0.0339 - accuracy: 0.9886 - val_

**Shallow Models**

In [117]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import joblib

models = {
    "DT": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(n_estimators=100, random_state=42),
    "ET": ExtraTreesClassifier(n_estimators=100, random_state=42),
    "XGB": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
}
results = []

for name, model in models.items():
    result_shallow = evaluate_shallow_model(name, model, X_train, y_train, X_test, y_test, save_path="models")
    results.append(result_shallow)

DT
RF
ET


/opt/miniconda3/envs/road/lib/python3.10/site-packages/xgboost/training.py:199: UserWarning: [17:16:40] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1768314084485/work/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGB


**Updating DNN Model**

In [118]:
# Updating DNN model
dnn_model.save("models/dnn_model.h5")
results.append(result_dnn)

## Print TABLE II

In [126]:
results_df = pd.DataFrame(results, columns=["Model", "Benign Samples", "Malicious Samples", "F1 Score", "FP", "FN", "MCC"])
results_df.to_csv("results/tableII.csv", index=False)
print(results_df)

  Model  Benign Samples  Malicious Samples  F1 Score   FP    FN       MCC
0    DT          450554              14899  0.910254  941  1668  0.907666
1    RF          450554              14899  0.910278  951  1659  0.907673
2    ET          450554              14899  0.910067  929  1683  0.907499
3   XGB          450554              14899  0.910595  966  1638  0.907967
4   DNN          450554              14899  0.842241  384  3781  0.845151


## LATEX TABLE II

In [124]:
df = pd.DataFrame(results, columns=["Model","Benign Samples","Malicious Samples","F1 Score","FP","FN","MCC"])

# reorder + drop F1 if you don’t want it
df = df[["Model","Benign Samples","Malicious Samples","FP","FN","MCC"]]

# format numbers with commas
for col in ["Benign Samples","Malicious Samples","FN","FP"]:
    df[col] = df[col].map(lambda x: f"{int(x):,}")

df["MCC"] = df["MCC"].map(lambda x: f"{x:.3f}")

# make the first row contain the Benign/Malicious values, blank others
df.loc[1:, "Benign Samples"] = ""
df.loc[1:, "Malicious Samples"] = ""

latex = df.to_latex(
    index=False,
    escape=False,
    column_format="|l|r|r|r|r|r|",
    caption="Performance Metrics on Test Samples under Benign Settings.",
    label="table:road_metrics"
)

# inject multirow (requires \\usepackage{multirow})
latex = latex.replace(
    df.iloc[0]["Benign Samples"],
    f"\\multirow{{5}}{{*}}{{{df.iloc[0]['Benign Samples']}}}"
).replace(
    df.iloc[0]["Malicious Samples"],
    f"\\multirow{{5}}{{*}}{{{df.iloc[0]['Malicious Samples']}}}"
)

print(latex)


\begin{table}
\caption{Performance Metrics on Test Samples under Benign Settings.}
\label{table:road_metrics}
\begin{tabular}{|l|r|r|r|r|r|}
\toprule
Model & Benign Samples & Malicious Samples & FP & FN & MCC \\
\midrule
DT & \multirow{5}{*}{450,554} & \multirow{5}{*}{14,899} & 941 & 1,668 & 0.908 \\
RF &  &  & 951 & 1,659 & 0.908 \\
ET &  &  & 929 & 1,683 & 0.907 \\
XGB &  &  & 966 & 1,638 & 0.908 \\
DNN &  &  & 384 & 3,781 & 0.845 \\
\bottomrule
\end{tabular}
\end{table}



In [ ]:
print("Train samples:", len(X_train), "Test samples:", len(X_test), "Total samples: ", len(X_test)+len(X_train))

Train samples: 1086055 Test samples: 465453 Total samples:  1551508
1.0 0.9679904969874471 0.032009503012552946
